<a href="https://colab.research.google.com/github/Jgour5454/Brain-Tumor-Detection-DL/blob/main/Yolo_v26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Yolo 26

In [ ]:
# ==============================================================================
# 1. SETUP & DATA EXTRACTION
# ==============================================================================
import os, zipfile, scipy.io, cv2, numpy as np, torch
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from google.colab import drive

# Mount Drive to access your zip
drive.mount('/content/drive', force_remount=True)

# Install Latest Ultralytics for YOLO26 support
!pip install -U -q "ultralytics>=8.4.0" scipy tqdm opencv-python-headless h5py

from ultralytics import YOLO

# Paths
ZIP_PATH = '/content/drive/MyDrive/ML/1512427.zip' # Path from your previous chats
RAW_EXTRACT = '/content/figshare_raw'
MAT_DIR = '/content/all_mats'
YOLO_ROOT = '/content/brain_tumor_yolo'

# Extract if not already done
if not os.path.exists(MAT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(RAW_EXTRACT)
    os.makedirs(MAT_DIR, exist_ok=True)
    for sz in [f for f in os.listdir(RAW_EXTRACT) if f.endswith('.zip')]:
        with zipfile.ZipFile(os.path.join(RAW_EXTRACT, sz), 'r') as z:
            z.extractall(MAT_DIR)

# ==============================================================================
# 2. CONVERT .MAT TO YOLO FORMAT
# ==============================================================================
for p in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(YOLO_ROOT, p), exist_ok=True)

processed_data = []
mat_files = [f for f in os.listdir(MAT_DIR) if f.endswith('.mat')]

for fname in tqdm(mat_files, desc="Converting Dataset"):
    try:
        # Load .mat (handles both v7.3 and older formats)
        try:
            m = scipy.io.loadmat(os.path.join(MAT_DIR, fname))['cjdata'][0][0]
            lbl, img, msk = int(m[0][0][0])-1, m[2].astype(np.float32), m[4]
        except:
            import h5py
            with h5py.File(os.path.join(MAT_DIR, fname), 'r') as f:
                c = f['cjdata']
                lbl, img, msk = int(c['label'][0][0])-1, np.array(c['image']).T, np.array(c['tumorMask']).T

        # Normalize Image
        img = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)

        # Get Bounding Box from Mask
        y, x = np.where(msk > 0)
        if len(x) > 0:
            h_i, w_i = img.shape
            xc, yc = (x.min() + x.max()) / 2 / w_i, (y.min() + y.max()) / 2 / h_i
            wn, hn = (x.max() - x.min()) / w_i, (y.max() - y.min()) / h_i
            processed_data.append((img, fname.replace('.mat', '.jpg'), f"{lbl} {xc} {yc} {wn} {hn}\n"))
    except: continue

# Split and Save
train_set, val_set = train_test_split(processed_data, test_size=0.2, random_state=42)
for data, subset in [(train_set, 'train'), (val_set, 'val')]:
    for im, nm, txt in data:
        cv2.imwrite(f'{YOLO_ROOT}/images/{subset}/{nm}', im)
        with open(f'{YOLO_ROOT}/labels/{subset}/{nm.replace(".jpg", ".txt")}', 'w') as f:
            f.write(txt)

# ==============================================================================
# 3. HIGH-SPEED YOLO26 TRAINING
# ==============================================================================
with open('brain.yaml', 'w') as f:
    f.write(f"path: {YOLO_ROOT}\ntrain: images/train\nval: images/val\nnames: [meningioma, glioma, pituitary]")

# Detect GPU
device = 0 if torch.cuda.is_available() else 'cpu'
print(f"🚀 Training on: {torch.cuda.get_device_name(0) if device==0 else 'CPU'}")

# Load YOLO26-Nano (Optimized for speed)
model = YOLO('yolo26n.pt')

model.train(
    data='brain.yaml',
    epochs=50,
    imgsz=640,
    batch=32,          # High batch size for faster GPU throughput
    device=device,
    amp=True,          # Uses Mixed Precision for 2x speedup
    optimizer='MuSGD', # New 2026 stable optimizer
    project='Brain_YOLO26',
    name='fast_run',
    exist_ok=True
)

# Backup results to Drive
!cp Brain_YOLO26/fast_run/weights/best.pt /content/drive/MyDrive/ML/yolo26_brain_best.pt
print("✅ Done! Weights saved to Drive.")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 69.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Converting Dataset: 100%|██████████| 3064/3064 [00:23<00:00, 130.97it/s]


🚀 Training on: Tesla T4
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=brain.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fast_run, nbs=64, nms=False, opset=None, optimize=False, optimizer=MuSGD, overlap_mask=True, patience=100, perspe